[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_12_Fine_Tuning_Fundamentals.ipynb)

# 🎓 Lesson 12: Fine-tuning Fundamentals
### *When to fine-tune, how to prep data, and LoRA/QLoRA in practice*

---

**Your Journey So Far:**
- ✅ Phase 1: LLMs → Prompt Engineering → Tool Use → Agents → Memory → Multi-Agent → RAG → Production → Capstone
- ✅ Phase 2, Lesson 10: Structured Outputs & Pydantic
- ✅ Phase 2, Lesson 11: LangGraph stateful graphs
- 🎯 **Today — Lesson 12:** Fine-tuning Fundamentals

**What You'll Learn:**
1. What fine-tuning is and **exactly when** it's the right tool (vs. prompting, RAG, or doing nothing)
2. How to prepare training data (the format that matters)
3. **LoRA / QLoRA** — the technique that makes fine-tuning affordable
4. Running a real fine-tuning job on a small model
5. How to evaluate whether fine-tuning actually helped

**⚡ GPU Recommendation:** Set your Colab runtime to T4 GPU (Runtime → Change runtime type → T4 GPU). The demo will still run on CPU but training will be *much* faster on GPU.

---

## 🧠 Part 1: What Is Fine-tuning?

A large language model like Claude or GPT-4 was trained in two phases:

```
Phase 1: Pre-training
  → Trained on ~trillions of tokens from the internet
  → Learns general language, reasoning, knowledge
  → Extremely expensive (millions of dollars, months of compute)
  → YOU NEVER DO THIS. Anthropic/OpenAI/Meta do this.

Phase 2: Fine-tuning (Instruction Tuning / RLHF)
  → Trained on curated examples: "do this task this way"
  → Teaches the model to be helpful, harmless, honest
  → Far cheaper. This is what turns a raw model into ChatGPT/Claude.
  → YOU CAN DO THIS on open-source models.
```

**Fine-tuning = starting from a pre-trained model and continuing to train it on YOUR data to specialize its behavior.**

Think of it like hiring a brilliant generalist (pre-trained model) and then giving them intensive training on YOUR company's specific processes, tone, and domain knowledge.

### The Three Levers for Improving Model Behavior

| Approach | What it does | Cost | When to use |
|----------|-------------|------|-------------|
| **Better Prompting** | Changes the input, not the model | ~Free | Always try first |
| **RAG** | Gives the model fresh/private facts at query time | Low | When knowledge is the gap |
| **Fine-tuning** | Changes model weights permanently | Medium-High | When *behavior* is the gap |

The biggest mistake engineers make is jumping straight to fine-tuning when better prompting or RAG would have solved the problem in an afternoon.

## 🎯 Part 2: The Decision Framework — Should You Fine-tune?

Use this checklist before spending ANY money on fine-tuning:

### ✅ Fine-tuning is the RIGHT choice when:

1. **Consistent format/style is required** — e.g., you need responses in a very specific JSON structure, and even with perfect prompting the model occasionally deviates
2. **Domain-specific knowledge that can't be RAG'd** — e.g., proprietary reasoning patterns, not just facts
3. **Latency or cost constraints** — a fine-tuned small model (7B params) can match a large model's quality on your specific task, at 10x lower cost
4. **Behavior change** — you want the model to always respond in a certain persona, language level, or style — RAG and prompting can't reliably enforce this
5. **You have 500+ high-quality examples** — fine-tuning with poor/insufficient data makes things *worse*

### ❌ Fine-tuning is the WRONG choice when:

1. You haven't tried prompt engineering yet (do that first!)
2. The problem is that the model lacks recent or private *facts* (use RAG)
3. You have fewer than ~100 examples (too little data)
4. Your examples are low quality or inconsistent (garbage in = garbage out)
5. You need the model to do something it fundamentally can't do (fine-tuning can't add new reasoning capabilities)

### The Decision Tree

```
Is the problem about KNOWLEDGE (facts, documents, up-to-date info)?
  → YES: Use RAG. Done.
  → NO: Continue...

Have you tried a well-crafted system prompt + few-shot examples?
  → NO: Do that first. It often works.
  → YES: Continue...

Is the issue about consistent FORMAT, STYLE, or BEHAVIOR?
  → YES + you have 500+ quality examples: Fine-tuning is your answer.
  → YES + fewer examples: Try more structured prompting first.
  → NO: Re-examine what the actual problem is.
```

**Real example:** A customer support bot that needs to always respond in a formal tone, cite a ticket number in every response, and never use bullet points — that's a behavior/format problem. After exhausting prompting, fine-tuning on 1000 examples of ideal responses would lock in this behavior reliably.

## 🔬 Part 3: LoRA — The Technique That Makes Fine-tuning Affordable

### The Problem with Full Fine-tuning

A model like Llama 3 (8B parameters) has **8 billion numbers** (weights). To fully fine-tune it, you'd update ALL 8 billion weights during training. That requires:
- Multiple high-end GPUs (A100s at ~$2/hour each)
- Days of training time
- Careful management of catastrophic forgetting

Not practical for most engineers.

### LoRA: Low-Rank Adaptation

**Key insight:** You don't need to update all 8B weights. Most of the "adaptation" needed for a specific task can be captured by training a small number of new weights and *adding* them to the frozen original weights.

Here's the math intuition (no PhD required):

```
Original weight matrix W:  (4096 × 4096) = 16,777,216 numbers
                           [FROZEN — never updated]

LoRA adds two small matrices:
  Matrix A:  (4096 × 8)   =    32,768 numbers  [trainable]
  Matrix B:  (8 × 4096)   =    32,768 numbers  [trainable]

During forward pass:  output = W·x + (B·A)·x · alpha/r
                               └──┘   └──────┘
                           frozen    LoRA adapter

Trainable parameters: 65,536 instead of 16,777,216
                      → 256x fewer parameters to train!
```

The **rank** (r=8 above) controls the capacity of the LoRA adapter:
- Low rank (r=4, r=8): Fewer parameters, faster, less expressive
- High rank (r=64, r=128): More capacity, can capture complex adaptations

For most tasks, **r=8 to r=64** is sufficient.

### QLoRA = LoRA + Quantization

**QLoRA** goes one step further: it loads the frozen base model in **4-bit quantization** (numbers stored in 4 bits instead of the usual 16 or 32 bits) to dramatically reduce memory, then trains LoRA adapters on top.

```
Model memory comparison (Llama 3 8B):
  Full precision (fp32):   ~32 GB VRAM
  Half precision (fp16):   ~16 GB VRAM  
  LoRA (fp16 base):        ~16 GB VRAM (base) + tiny LoRA adapters
  QLoRA (4-bit base):       ~5 GB VRAM  ← Fits on a FREE Colab T4!
```

**QLoRA is the technique you'll use in 90% of real fine-tuning projects.** It lets you fine-tune multi-billion parameter models on a single consumer GPU.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# Runtime: ~2-3 minutes on first run
# ============================================================

!pip install transformers datasets peft accelerate bitsandbytes trl -q
!pip install torch --quiet  # Usually pre-installed in Colab

print("✅ All packages installed!")

In [ ]:
# ============================================================
# CELL 2: Check Your Hardware
# ============================================================

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {gpu_memory:.1f} GB")
    print("\n✅ GPU found! You'll get fast training.")
else:
    print("\n⚠️  No GPU detected. Training will be slow.")
    print("   Tip: Runtime → Change runtime type → T4 GPU")
    print("   The demo will still run for learning purposes!")

## 📦 Part 4: Data Preparation

This is often the most important (and most overlooked) part of fine-tuning.

### The Golden Rule of Fine-tuning Data
> **Quality beats quantity every time.** 200 perfect examples will outperform 10,000 mediocre ones.

### The Standard Format: Instruction Tuning

Most modern fine-tuning uses the **instruction-response** format. Every training example is a conversation:

```json
{
  "messages": [
    {"role": "system",    "content": "You are a helpful assistant that..."},
    {"role": "user",      "content": "The user's question or task"},
    {"role": "assistant", "content": "The ideal response you want the model to produce"}
  ]
}
```

This format works for: OpenAI fine-tuning API, HuggingFace TRL (SFTTrainer), Anthropic fine-tuning API.

### What Makes a Good Training Example?

1. **The assistant response is exactly what you want** — not "good enough", but ideal
2. **Diverse inputs** — cover the range of real user queries, not just easy ones
3. **Consistent style** — if 500 examples use formal tone, make sure ALL do
4. **No contradictions** — if example #1 says "always use bullet points" and example #200 never does, the model will be confused

### How Many Examples?

| Task complexity | Minimum examples | Sweet spot |
|----------------|-----------------|------------|
| Simple format change | 100-200 | 500 |
| Domain tone/style | 300-500 | 1,000+ |
| Complex task behaviors | 1,000+ | 5,000+ |
| Full domain adaptation | 5,000+ | 10,000+ |

In [ ]:
# ============================================================
# CELL 3: Build a Training Dataset
# 
# Scenario: Fine-tuning a model to answer questions about
# Python programming in a concise, code-focused style.
# ============================================================

from datasets import Dataset
import json

# Our training examples in the ChatML / messages format
# In real life, you'd have hundreds/thousands of these
SYSTEM_PROMPT = (
    "You are a Python expert. Answer concisely with working code examples. "
    "Always include a brief explanation after the code."
)

raw_data = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How do I reverse a list in Python?"},
            {"role": "assistant", "content": "```python\nmy_list = [1, 2, 3, 4, 5]\nreversed_list = my_list[::-1]  # slicing\n# or: my_list.reverse()  # in-place\n```\nUse `[::-1]` for a new reversed list, or `.reverse()` to modify in-place."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How do I read a JSON file in Python?"},
            {"role": "assistant", "content": "```python\nimport json\n\nwith open('data.json', 'r') as f:\n    data = json.load(f)\n\nprint(data)  # dict or list\n```\n`json.load()` reads from a file object. Use `json.loads()` for a JSON string."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "What's the difference between a list and a tuple?"},
            {"role": "assistant", "content": "```python\nmy_list = [1, 2, 3]   # mutable — can change\nmy_tuple = (1, 2, 3)  # immutable — cannot change\n\nmy_list[0] = 99    # ✅ works\n# my_tuple[0] = 99  # ❌ TypeError\n```\nUse tuples for data that shouldn't change (coordinates, RGB values). Lists for collections you'll modify."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How do I handle exceptions in Python?"},
            {"role": "assistant", "content": "```python\ntry:\n    result = 10 / 0\nexcept ZeroDivisionError as e:\n    print(f'Error: {e}')\nexcept Exception as e:\n    print(f'Unexpected error: {e}')\nelse:\n    print('Success:', result)  # runs if no exception\nfinally:\n    print('Always runs')  # cleanup code\n```\nCatch specific exceptions before general ones. Use `finally` for cleanup (closing files, DB connections)."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How do I use list comprehension?"},
            {"role": "assistant", "content": "```python\n# Pattern: [expression for item in iterable if condition]\n\nsquares = [x**2 for x in range(10)]\n# [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]\n\nevens = [x for x in range(20) if x % 2 == 0]\n# [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]\n\n# Nested (use sparingly — readability matters)\nmatrix = [[i*j for j in range(3)] for i in range(3)]\n```\nList comprehensions are faster than equivalent `for` loops and more Pythonic. Avoid nesting more than 2 levels."}
        ]
    },
]

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(raw_data)

print(f"📊 Dataset size: {len(dataset)} examples")
print(f"\n📝 Example 0 preview:")
print(f"  User: {dataset[0]['messages'][1]['content']}")
print(f"  Assistant: {dataset[0]['messages'][2]['content'][:80]}...")

In [ ]:
# ============================================================
# CELL 4: Format Data for Training (Apply Chat Template)
#
# Models expect data in a specific text format called a
# 'chat template'. Different models use different formats.
# HuggingFace handles this automatically via the tokenizer.
# ============================================================

from transformers import AutoTokenizer

# We'll use a small model for this demo
# For real fine-tuning, replace with: "meta-llama/Llama-3.2-1B-Instruct"
MODEL_NAME = "microsoft/phi-2"  # 2.7B param model, good for demo
# 💡 EXPERIMENT: Try "TinyLlama/TinyLlama-1.1B-Chat-v1.0" for even faster loading

print(f"⏳ Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Add pad token if missing (needed for batched training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ Tokenizer loaded!")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")
print(f"   Max length: {tokenizer.model_max_length}")

# Show how a message looks after tokenization
sample_text = "How do I reverse a list?"
tokens = tokenizer.encode(sample_text)
print(f"\n📝 Sample tokenization:")
print(f"   Text: '{sample_text}'")
print(f"   Tokens ({len(tokens)}): {tokens}")
print(f"   Decoded back: {tokenizer.decode(tokens)}")

## 🔧 Part 5: Setting Up LoRA with PEFT

PEFT (Parameter-Efficient Fine-Tuning) is the HuggingFace library that implements LoRA, QLoRA, and other efficient fine-tuning methods.

The key configuration choices you'll make:

| Parameter | What it does | Typical values |
|-----------|-------------|----------------|
| `r` | LoRA rank — controls adapter capacity | 8, 16, 32, 64 |
| `lora_alpha` | Scaling factor (set to 2×r as a default) | 16, 32, 64 |
| `lora_dropout` | Regularization (prevents overfitting) | 0.05-0.1 |
| `target_modules` | Which layers to apply LoRA to | "all-linear" or specific names |
| `task_type` | Type of fine-tuning task | CAUSAL_LM, SEQ_CLS |

**Rule of thumb:** Start with `r=16`, `lora_alpha=32`, `lora_dropout=0.05`. Increase `r` if the model isn't adapting enough.

In [ ]:
# ============================================================
# CELL 5: Load Model + Apply LoRA Configuration
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch

print(f"⏳ Loading model {MODEL_NAME}...")
print(f"   Device: {device}")

# --- Quantization config (QLoRA) ---
# This loads the base model in 4-bit to save VRAM
if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                          # 4-bit quantization
        bnb_4bit_use_double_quant=True,             # nested quantization (saves more VRAM)
        bnb_4bit_quant_type="nf4",                  # NormalFloat4 — best for LLM weights
        bnb_4bit_compute_dtype=torch.bfloat16,      # compute in bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    # Prepare for k-bit training (adds gradient checkpointing, etc.)
    model = prepare_model_for_kbit_training(model)
    print("   ✅ Loaded in 4-bit (QLoRA mode)")
else:
    # CPU mode: load in float32 (smaller model needed for CPU)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True
    )
    print("   ✅ Loaded in float32 (CPU mode — training will be slow)")

# --- LoRA Configuration ---
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # We're doing causal language modeling
    r=16,                           # Rank — adapter capacity
    lora_alpha=32,                  # Scaling factor (typically 2×r)
    lora_dropout=0.05,              # Small dropout for regularization
    target_modules="all-linear",    # Apply LoRA to all linear layers
    bias="none",                    # Don't train bias terms
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Show parameter counts
model.print_trainable_parameters()

In [ ]:
# ============================================================
# CELL 6: Configure Training with SFTTrainer (TRL)
#
# TRL (Transformer Reinforcement Learning) library provides
# SFTTrainer — a HuggingFace Trainer optimized for
# Supervised Fine-Tuning (SFT) of LLMs.
# ============================================================

from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Training configuration
training_args = SFTConfig(
    # Output
    output_dir="./lora-python-expert",
    
    # Training loop
    num_train_epochs=3,              # 3 passes over the data
    per_device_train_batch_size=1,   # 1 example at a time (small model/memory)
    gradient_accumulation_steps=4,  # Simulate batch_size=4 by accumulating gradients
    
    # Optimizer
    learning_rate=2e-4,              # Standard LoRA learning rate
    warmup_steps=5,                  # Gradual LR warmup at start
    weight_decay=0.01,               # L2 regularization
    optim="adamw_torch",             # AdamW optimizer
    
    # Precision
    fp16=(device == "cuda"),         # Mixed precision on GPU
    bf16=False,                      # Use fp16 instead of bf16 (T4 compatibility)
    
    # Logging
    logging_steps=1,                 # Log every step
    report_to="none",                # Don't send to W&B / TensorBoard
    
    # Sequence length
    max_seq_length=512,              # Max tokens per training example
)

# 💡 EXPERIMENT: Try different learning rates (1e-4, 5e-4)
# and observe how fast/slow the loss drops

# Initialize the trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

print("✅ Trainer initialized!")
print(f"   Training examples: {len(dataset)}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")

In [ ]:
# ============================================================
# CELL 7: Run Training! 🚀
#
# Watch the 'loss' metric as training progresses:
#   - Loss starts high (model is random/unpredictable)
#   - Loss should decrease with each step (model is learning)
#   - If loss plateaus or increases: LR too high, data issue, or overfitting
# ============================================================

print("🚀 Starting training...")
print("   Watch the loss decrease as the model learns your data!\n")

train_result = trainer.train()

# Print final training metrics
print(f"\n{'='*50}")
print("✅ Training Complete!")
print(f"   Final loss: {train_result.training_loss:.4f}")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training time: {train_result.metrics.get('train_runtime', 0):.1f}s")

# Understanding the loss:
# Loss < 1.0: Model is learning well
# Loss 0.3-0.7: Good convergence  
# Loss < 0.1: Potential overfitting (memorizing, not generalizing)

In [ ]:
# ============================================================
# CELL 8: Save the Fine-tuned LoRA Adapter
#
# KEY INSIGHT: We only save the SMALL LoRA adapter weights,
# not the entire model. The base model stays unchanged.
# ============================================================

ADAPTER_PATH = "./python-expert-adapter"

# Save only the LoRA adapter (not the full model)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

import os
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
) / 1e6

print(f"✅ LoRA adapter saved to: {ADAPTER_PATH}")
print(f"   Adapter size: {adapter_size:.1f} MB")
print(f"   (The full model would be much larger — only the adapter is saved!)")
print()
print("📁 Files saved:")
for f in os.listdir(ADAPTER_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f)) / 1e6
    print(f"   {f}: {size:.2f} MB")

## 📊 Part 6: Evaluation — Did Fine-tuning Actually Help?

Fine-tuning without evaluation is flying blind. You need to measure whether the model improved.

### Evaluation Methods

**1. Perplexity** — How "surprised" is the model by your test data?
- Lower perplexity = model finds your data more predictable = better alignment with your style
- Compare: base model perplexity vs. fine-tuned model perplexity on held-out examples

**2. Side-by-side comparison** — Generate responses from both base and fine-tuned models, compare manually or with an LLM judge

**3. Task-specific metrics** — If you're fine-tuning for classification, measure accuracy. For summarization, measure ROUGE score. For code, run unit tests.

**4. Overfitting check** — If train loss is low but test loss is high, the model memorized training data rather than learning generalizable patterns.

In [ ]:
# ============================================================
# CELL 9: Side-by-Side Evaluation
# Compare base model vs. fine-tuned model responses
# ============================================================

from peft import PeftModel
import torch

def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    """Generate a response from the model."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    if device == "cuda":
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the newly generated tokens (not the input)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test question (something the model was NOT trained on explicitly)
test_question = "How do I sort a dictionary by value in Python?"

# Create the prompt in the model's expected format
test_prompt = f"Instruction: {test_question}\nResponse:"

print("🧪 Evaluation Question:", test_question)
print("=" * 60)

# Test the fine-tuned model
print("\n🎯 Fine-tuned Model Response:")
finetuned_response = generate_response(model, tokenizer, test_prompt)
print(finetuned_response)

print("\n" + "=" * 60)
print("💡 What to look for:")
print("   - Does it include a code block?")
print("   - Is there a brief explanation after the code?")
print("   - Is the code correct and concise?")
print("   These are qualities we trained it to have!")

In [ ]:
# ============================================================
# CELL 10: LLM-as-Judge Evaluation (Using Claude API)
#
# A powerful technique: use a stronger model to evaluate
# your fine-tuned model's outputs automatically.
# ============================================================

import anthropic
from google.colab import userdata

try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
    client = anthropic.Anthropic(api_key=api_key)
    
    # The response from our fine-tuned model
    finetuned_output = generate_response(model, tokenizer, test_prompt, max_new_tokens=150)
    
    JUDGE_PROMPT = f"""You are evaluating a Python Q&A response. Score it from 1-5 on three criteria.

Question: {test_question}

Response to evaluate:
{finetuned_output}

Evaluate on:
1. CODE_QUALITY (1-5): Is the code correct and Pythonic?
2. FORMAT (1-5): Does it include both code AND a brief explanation?
3. CONCISENESS (1-5): Is it concise without unnecessary padding?

Output as JSON: {{"code_quality": X, "format": X, "conciseness": X, "comment": "one line"}}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Use Haiku for cost efficiency in eval
        max_tokens=200,
        messages=[{"role": "user", "content": JUDGE_PROMPT}]
    )
    
    import json
    scores = json.loads(response.content[0].text)
    print("🏆 LLM Judge Scores:")
    print(f"   Code Quality:  {'⭐' * scores['code_quality']} ({scores['code_quality']}/5)")
    print(f"   Format:        {'⭐' * scores['format']} ({scores['format']}/5)")
    print(f"   Conciseness:   {'⭐' * scores['conciseness']} ({scores['conciseness']}/5)")
    print(f"   Judge comment: {scores['comment']}")
    
except Exception as e:
    print(f"⚠️  LLM judge skipped: {e}")
    print("   Add ANTHROPIC_API_KEY to Colab Secrets to enable automated evaluation.")
    print("   (This cell is optional — manual evaluation also works!)") 

# 💡 EXPERIMENT: Run this judge on 10+ questions and compute average scores.
# Compare to the base model's average scores to measure fine-tuning uplift.

## 🚀 Part 7: Advanced Fine-tuning Concepts

Now that you understand SFT (Supervised Fine-Tuning), here are the techniques that take it further:

### RLHF — Reinforcement Learning from Human Feedback
What turned GPT-3 into ChatGPT. Three stages:
1. **SFT** (what we did above) — train on ideal responses
2. **Reward Model Training** — train a model to predict which responses humans prefer  
3. **PPO/REINFORCE** — use RL to optimize the language model to maximize the reward model's score

RLHF is expensive and complex. Most engineers use DPO instead.

### DPO — Direct Preference Optimization
A simpler alternative to RLHF. Instead of training a separate reward model, DPO trains directly on **preference pairs**:
```json
{
  "prompt": "How should I handle errors in Python?",
  "chosen":  "Use try/except with specific exception types...",   // preferred
  "rejected": "Just use a bare except clause for everything..."   // not preferred
}
```
DPO is what most modern fine-tuning pipelines use for alignment. TRL has a `DPOTrainer` that works just like `SFTTrainer`.

### Merging LoRA Adapters
After training, you can **merge** the LoRA adapter back into the base model weights:
```python
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged-model")
```
This creates a single standalone model (no adapter overhead) — useful for deployment.

### Cloud API Fine-tuning (No GPU Required)
If you don't want to manage GPUs:
- **OpenAI Fine-tuning API** — supports GPT-4o-mini and GPT-3.5, JSONL format, ~$8/1M training tokens
- **Anthropic Fine-tuning** — available via API for Claude models (enterprise tier)
- **Google Vertex AI** — Gemini fine-tuning

These are production-grade, no infrastructure to manage, but cost more per training run.

In [ ]:
# ============================================================
# CELL 11: Merge LoRA Adapter into Base Model
# ============================================================

print("🔀 Merging LoRA adapter into base model weights...")

# This combines: base_weights + LoRA_A × LoRA_B
# The result is a standalone model with no adapter overhead
merged_model = model.merge_and_unload()

print("✅ Adapter merged!")
print()
print("📊 Before merge:")
print("   model = base_model + LoRA_adapter (separate files)")
print()
print("📊 After merge:")
print("   merged_model = one unified model (no adapter overhead)")
print()
print("💡 Use merging when you want to deploy the model for inference.")
print("   Keep the adapter separate during development so you can iterate.")

# Save the merged model (optional — it's large)
# merged_model.save_pretrained("./merged-python-expert")
# print("✅ Merged model saved!")

In [ ]:
# ============================================================
# CELL 12: Cloud Fine-tuning via OpenAI API (Conceptual)
#
# This shows the API-based approach — no GPU needed.
# Uncomment and run with a real OpenAI key to execute.
# ============================================================

# CONCEPT DEMO — shows the API structure
# Same data format (ChatML messages) works across providers!

import json

# 1. Prepare data as JSONL (one JSON object per line)
jsonl_content = []
for example in raw_data:
    jsonl_content.append(json.dumps(example))

print("📄 JSONL format (what you'd upload to OpenAI/Anthropic):")
print("=" * 60)
for line in jsonl_content[:2]:  # Show first 2 examples
    print(line[:120] + "...")
print(f"... ({len(jsonl_content)} total lines)")

# Save to file
with open("/tmp/training_data.jsonl", "w") as f:
    f.write("\n".join(jsonl_content))

print(f"\n✅ Saved to training_data.jsonl")
print()
print("📋 OpenAI Fine-tuning API Steps:")
print("   1. client.files.create(file=open('training_data.jsonl'))")
print("   2. client.fine_tuning.jobs.create(model='gpt-4o-mini', training_file=file_id)")
print("   3. Poll job status: client.fine_tuning.jobs.retrieve(job_id)")
print("   4. Use: client.chat.completions.create(model='ft:gpt-4o-mini:...')")
print()
print("💰 Cost estimate for this dataset:")
total_chars = sum(len(json.dumps(ex)) for ex in raw_data)
est_tokens = total_chars // 4  # rough estimate
print(f"   ~{est_tokens:,} training tokens")
print(f"   At $3/1M tokens × 3 epochs ≈ ${est_tokens * 3 / 1e6 * 3:.4f}")
print(f"   (Very cheap for small datasets!)")  

## 🏎️ Bonus: Unsloth — 2x Faster LoRA Training

For production fine-tuning, **[Unsloth](https://github.com/unslothai/unsloth)** is the go-to library. It's a drop-in replacement for HuggingFace PEFT that trains 2x faster with 60% less VRAM.

```python
# Unsloth replaces the model loading + LoRA setup in ~5 lines:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",  # optimized model
    max_seq_length=2048,
    load_in_4bit=True,  # QLoRA automatically
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"],
)

# Then use SFTTrainer exactly as above — it's compatible!
```

**When to use Unsloth:** When fine-tuning real models (3B+ params) on Colab T4/A100. It stays within the free GPU tier memory limits better than vanilla PEFT.

## 📚 Lesson 12 Recap

### What You Learned

**1. The Decision Framework**
- Try prompting first → then RAG → then fine-tuning
- Fine-tune when: consistent behavior/format is required AND you have quality data
- Fine-tuning does NOT add new reasoning capabilities — only adapts behavior

**2. LoRA / QLoRA**
- LoRA trains two small matrices (A, B) instead of updating all model weights
- QLoRA = 4-bit base model + LoRA adapters → fits a 7B model on a T4 GPU
- Rank (r) controls capacity; start at r=16, tune from there

**3. Data Preparation**
- Instruction-response format (ChatML messages) is the standard
- Quality > Quantity: 500 perfect examples beat 10K mediocre ones
- Consistency is critical — contradictory examples confuse the model

**4. The Training Pipeline**
- `SFTTrainer` (TRL) handles the training loop
- Watch the loss: decreasing = learning; plateauing = try lower LR; < 0.1 = overfitting risk
- Save just the LoRA adapter (small); merge into base model for deployment

**5. Evaluation**
- LLM-as-judge is a scalable automated eval method
- Always compare base model vs. fine-tuned model on held-out test questions

---

### 🔜 What's Next: Lesson 13 — Multimodal AI (Vision + Text)

You've been working exclusively with text. But Claude (and many modern LLMs) can see images. In Lesson 13, you'll:
- Send images to Claude and extract structured information from them
- Build a visual document parser (receipts, charts, diagrams)
- Combine vision + your existing RAG pipeline
- Explore video understanding with frame extraction

---

### 🛠️ Exercises to Solidify Your Learning

1. **Expand the dataset** — Add 20 more Python Q&A examples and re-run training. Does quality improve?
2. **Try different ranks** — Run with r=4, r=16, r=64. Compare final loss and response quality.
3. **Domain swap** — Change the training data to a different domain (SQL queries, regex patterns). Does the same pipeline work?
4. **Overfitting experiment** — Train for 20 epochs instead of 3. Watch the loss go to near-zero. Then test on new questions — does quality improve or degrade?
5. **DPO dataset** — Create 5 preference pairs (chosen vs. rejected responses) and look at the TRL `DPOTrainer` documentation.

---

*Lesson 12 of 15 — Fine-tuning Fundamentals | Generated 2026-05-11*